# Tree-based models for daily temperature regression

Notebook complementare a `baselines.ipynb`: mantiene **lo stesso dataset, le stesse feature, lo stesso orizzonte di previsione (`h`) e lo stesso split temporale** del notebook originale, per confrontare i modelli in modo corretto.

Modelli inclusi:
- **Decision Tree** — singolo albero;
- **Random Forest** — bagging di alberi con campionamento bootstrap e selezione casuale delle feature;
- **Extra Trees** — ensemble con maggiore casualità negli split;
- **Bagging Regressor** — bagging esplicito di alberi;
- **AdaBoost** — boosting sequenziale;
- **Gradient Boosting** e **HistGradientBoosting** — boosting basato sul gradiente.

La selezione degli iperparametri usa solo il validation set. Il test set viene usato per la valutazione finale dei modelli selezionati. Le metriche principali sono MAE, RMSE e R².

In [ ]:
# Setup: stesso ambiente e stessa struttura del progetto di baselines.ipynb
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    GITHUB_USER = "SinghSahiljit44"
    REPO_NAME = "Weather_forecasting"
    REPO_URL = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"

    if not os.path.exists(REPO_NAME):
        !git clone {REPO_URL}
    os.chdir(REPO_NAME)
    PROJECT_ROOT = Path.cwd()
else:
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print(f"Ambiente: {'Google Colab' if IN_COLAB else 'Locale'}")
print(f"Project root: {PROJECT_ROOT}")

## 1. Dataset, feature engineering e split temporale

Riprendiamo le feature del notebook di baseline: variabili meteorologiche giornaliere, codifica ciclica del giorno dell'anno, lag della temperatura e medie mobili. `Split` gestisce la costruzione del target futuro e la suddivisione temporale. Non facciamo uno split casuale, perché si tratta di una serie temporale.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from src.data.preprocessing import load_daily
from src.utils.split import Split

data = load_daily()

doy = data.index.dayofyear
data["doy_sin"] = np.sin(2 * np.pi * doy / 365.25)
data["doy_cos"] = np.cos(2 * np.pi * doy / 365.25)

for k in range(1, 4):
    data[f"t2m_lag{k}"] = data["t2m_mean"].shift(k)
data["t2m_roll7"] = data["t2m_mean"].rolling(7).mean()
data["t2m_roll30"] = data["t2m_mean"].rolling(30).mean()

features = [
    "t2m_mean", "t2m_min", "t2m_max", "tp_mm", "e_mm", "pev_mm",
    "skt_mean", "stl1_mean", "stl2_mean", "stl3_mean", "stl4_mean",
    "swvl1_mean", "swvl2_mean", "swvl3_mean", "swvl4_mean",
    "doy_sin", "doy_cos", "t2m_lag1", "t2m_lag2", "t2m_lag3",
    "t2m_roll7", "t2m_roll30"
]

h = 1  # giorni in avanti, come in baselines.ipynb
split = Split(data, "t2m_mean")
(X_train, y_train), (X_val, y_val), (X_test, y_test) = split.get(features, h)

print(f"Training:   {len(X_train):,} campioni")
print(f"Validation: {len(X_val):,} campioni")
print(f"Test:       {len(X_test):,} campioni")
print(f"Feature:    {len(features)}")
print(f"Target: temperatura media a {h} giorno/i di distanza")

## 2. Metriche e baseline

- **MAE**: errore assoluto medio, espresso in °C; più basso è meglio.
- **RMSE**: penalizza maggiormente gli errori grandi; più basso è meglio.
- **R²**: quota di variabilità spiegata dal modello; più alto è meglio.

Come riferimento includiamo una regressione lineare, usando lo stesso split.

In [ ]:
from sklearn.linear_model import LinearRegression

def regression_metrics(y_true, y_pred):
    return {
        "MAE (°C)": mean_absolute_error(y_true, y_pred),
        "RMSE (°C)": mean_squared_error(y_true, y_pred) ** 0.5,
        "R²": r2_score(y_true, y_pred),
    }

models = {}
validation_predictions = {}
test_predictions = {}
validation_rows = []

linear = LinearRegression()
linear.fit(X_train, y_train)
models["Linear Regression"] = linear
validation_predictions["Linear Regression"] = linear.predict(X_val)

validation_rows.append({
    "Model": "Linear Regression",
    **regression_metrics(y_val, validation_predictions["Linear Regression"])
})

display(pd.DataFrame(validation_rows).set_index("Model").round(4))

## 3. Decision Tree

Un singolo albero divide lo spazio delle feature in regioni e predice un valore costante in ogni regione. È interpretabile, ma può essere instabile e sovradattarsi: `max_depth`, `min_samples_leaf` e `min_samples_split` controllano la complessità.

Proviamo una griglia contenuta e scegliamo la configurazione con MAE minimo sul validation set.

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.base import clone

tree_candidates = [
    {"max_depth": depth, "min_samples_leaf": leaf, "min_samples_split": split_min}
    for depth in [3, 5, 8, 12, None]
    for leaf in [1, 5, 15]
    for split_min in [2, 10]
]

tree_trials = []
for params in tree_candidates:
    model = DecisionTreeRegressor(random_state=42, **params)
    model.fit(X_train, y_train)
    pred = model.predict(X_val)
    tree_trials.append({**params, **{"Val MAE": mean_absolute_error(y_val, pred)}})

tree_table = pd.DataFrame(tree_trials).sort_values("Val MAE").reset_index(drop=True)
display(tree_table.head(10))

best_tree_params = tree_table.iloc[0][["max_depth", "min_samples_leaf", "min_samples_split"]].to_dict()
# Pandas può convertire None in NaN; ripristiniamo il valore per max_depth se necessario.
if pd.isna(best_tree_params["max_depth"]):
    best_tree_params["max_depth"] = None
best_tree_params["min_samples_leaf"] = int(best_tree_params["min_samples_leaf"])
best_tree_params["min_samples_split"] = int(best_tree_params["min_samples_split"])

best_tree = DecisionTreeRegressor(random_state=42, **best_tree_params)
best_tree.fit(X_train, y_train)
models["Decision Tree"] = best_tree
validation_predictions["Decision Tree"] = best_tree.predict(X_val)

print("Best parameters:", best_tree_params)
print("Validation metrics:", regression_metrics(y_val, validation_predictions["Decision Tree"]))

## 4. Bagging e Random Forest

Il **bagging** addestra più modelli su campioni bootstrap e combina le loro predizioni, riducendo la varianza rispetto a un singolo albero. La **Random Forest** aggiunge casualità nella scelta delle feature considerate a ogni split. **Extra Trees** introduce ulteriore casualità nella scelta delle soglie di split.

Per contenere il tempo di esecuzione, testiamo poche configurazioni rappresentative. Tutte le scelte sono effettuate in base al validation MAE.

In [ ]:
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor, ExtraTreesRegressor

ensemble_candidates = {
    "Bagging": [
        {"n_estimators": n, "max_samples": sample, "max_features": 1.0, "max_depth": depth, "min_samples_leaf": leaf}
        for n in [100, 250]
        for sample in [0.7, 1.0]
        for depth, leaf in [(None, 2), (12, 5)]
    ],
    "Random Forest": [
        {"n_estimators": n, "max_features": mf, "max_depth": depth, "min_samples_leaf": leaf}
        for n in [150, 300]
        for mf in [0.7, 1.0]
        for depth, leaf in [(None, 2), (12, 5)]
    ],
    "Extra Trees": [
        {"n_estimators": n, "max_features": mf, "max_depth": depth, "min_samples_leaf": leaf}
        for n in [150, 300]
        for mf in [0.7, 1.0]
        for depth, leaf in [(None, 2), (12, 5)]
    ],
}

def make_ensemble(name, params):
    if name == "Bagging":
        # API sklearn recente: estimator=. Per versioni molto vecchie può essere base_estimator=.
        base_tree = DecisionTreeRegressor(
            max_depth=params["max_depth"],
            min_samples_leaf=params["min_samples_leaf"],
            random_state=42,
        )
        return BaggingRegressor(
            estimator=base_tree,
            n_estimators=params["n_estimators"],
            max_samples=params["max_samples"],
            max_features=params["max_features"],
            random_state=42,
            n_jobs=-1,
        )
    if name == "Random Forest":
        return RandomForestRegressor(random_state=42, n_jobs=-1, **params)
    return ExtraTreesRegressor(random_state=42, n_jobs=-1, **params)

ensemble_tables = {}
best_ensemble_params = {}

for name, candidates in ensemble_candidates.items():
    rows = []
    for params in candidates:
        model = make_ensemble(name, params)
        model.fit(X_train, y_train)
        pred = model.predict(X_val)
        rows.append({**params, "Val MAE": mean_absolute_error(y_val, pred)})
    table = pd.DataFrame(rows).sort_values("Val MAE").reset_index(drop=True)
    ensemble_tables[name] = table
    display(pd.DataFrame({name: table.head(5).to_dict(orient="records")}))
    best_ensemble_params[name] = {k: table.iloc[0][k] for k in candidates[0].keys()}
    for key in ["n_estimators", "min_samples_leaf"]:
        best_ensemble_params[name][key] = int(best_ensemble_params[name][key])
    if name == "Bagging":
        best_ensemble_params[name]["max_samples"] = float(best_ensemble_params[name]["max_samples"])
        best_ensemble_params[name]["max_features"] = float(best_ensemble_params[name]["max_features"])
    else:
        best_ensemble_params[name]["max_features"] = float(best_ensemble_params[name]["max_features"])
    if pd.isna(best_ensemble_params[name]["max_depth"]):
        best_ensemble_params[name]["max_depth"] = None
    else:
        best_ensemble_params[name]["max_depth"] = int(best_ensemble_params[name]["max_depth"])

    fitted = make_ensemble(name, best_ensemble_params[name])
    fitted.fit(X_train, y_train)
    models[name] = fitted
    validation_predictions[name] = fitted.predict(X_val)
    print(f"{name} — best params: {best_ensemble_params[name]}")
    print(f"Validation metrics: {regression_metrics(y_val, validation_predictions[name])}\n")

## 5. Boosting: AdaBoost, Gradient Boosting e HistGradientBoosting

Nel **boosting**, i modelli vengono costruiti in sequenza: ogni passaggio cerca di migliorare gli errori residui o la funzione obiettivo del modello corrente. A differenza del bagging, non si tratta semplicemente di addestrare alberi indipendenti in parallelo.

- **AdaBoost** aumenta l'attenzione sui campioni difficili.
- **Gradient Boosting** aggiunge alberi che approssimano il gradiente della loss.
- **HistGradientBoosting** discretizza le feature in bin e può essere più efficiente su dataset grandi.

Il learning rate controlla il contributo di ciascun albero; profondità e numero di alberi controllano la capacità del modello.

In [ ]:
from sklearn.ensemble import AdaBoostRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor

boosting_candidates = {
    "AdaBoost": [
        {"n_estimators": n, "learning_rate": lr, "max_depth": depth, "min_samples_leaf": leaf}
        for n in [100, 250]
        for lr in [0.03, 0.1, 0.3]
        for depth, leaf in [(2, 5), (3, 10)]
    ],
    "Gradient Boosting": [
        {"n_estimators": n, "learning_rate": lr, "max_depth": depth, "min_samples_leaf": leaf, "subsample": subsample}
        for n in [100, 250]
        for lr in [0.03, 0.1]
        for depth, leaf, subsample in [(2, 5, 0.8), (3, 10, 1.0)]
    ],
    "HistGradientBoosting": [
        {"max_iter": n, "learning_rate": lr, "max_leaf_nodes": leaves, "l2_regularization": reg}
        for n in [100, 250]
        for lr in [0.03, 0.1]
        for leaves in [15, 31]
        for reg in [0.0, 1.0]
    ],
}

def make_boosting(name, params):
    if name == "AdaBoost":
        tree = DecisionTreeRegressor(
            max_depth=params["max_depth"],
            min_samples_leaf=params["min_samples_leaf"],
            random_state=42,
        )
        # AdaBoostRegressor usa estimator nelle versioni recenti di scikit-learn.
        return AdaBoostRegressor(
            estimator=tree,
            n_estimators=params["n_estimators"],
            learning_rate=params["learning_rate"],
            random_state=42,
        )
    if name == "Gradient Boosting":
        return GradientBoostingRegressor(random_state=42, **params)
    return HistGradientBoostingRegressor(random_state=42, **params)

boosting_tables = {}
best_boosting_params = {}

for name, candidates in boosting_candidates.items():
    rows = []
    for params in candidates:
        model = make_boosting(name, params)
        model.fit(X_train, y_train)
        pred = model.predict(X_val)
        rows.append({**params, "Val MAE": mean_absolute_error(y_val, pred)})
    table = pd.DataFrame(rows).sort_values("Val MAE").reset_index(drop=True)
    boosting_tables[name] = table
    display(table.head(5))
    best = {k: table.iloc[0][k] for k in candidates[0].keys()}
    for key in ["n_estimators", "max_iter", "max_depth", "min_samples_leaf", "max_leaf_nodes"]:
        if key in best and pd.notna(best[key]):
            best[key] = int(best[key])
    for key in ["learning_rate", "subsample", "l2_regularization"]:
        if key in best:
            best[key] = float(best[key])
    if "l2_regularization" in best:
        best["l2_regularization"] = float(best["l2_regularization"])
    best_boosting_params[name] = best

    fitted = make_boosting(name, best)
    fitted.fit(X_train, y_train)
    models[name] = fitted
    validation_predictions[name] = fitted.predict(X_val)
    print(f"{name} — best params: {best}")
    print(f"Validation metrics: {regression_metrics(y_val, validation_predictions[name])}\n")

## 6. Confronto sul validation set

Questo confronto serve a scegliere i modelli candidati. Il test set non viene usato per scegliere gli iperparametri. MAE e RMSE sono in °C; un R² più alto è preferibile, ma può essere negativo se il modello è peggiore della baseline che predice la media del target.

In [ ]:
validation_rows = []
for name, pred in validation_predictions.items():
    validation_rows.append({"Model": name, **regression_metrics(y_val, pred)})

validation_results = (
    pd.DataFrame(validation_rows)
    .sort_values("MAE (°C)")
    .reset_index(drop=True)
)
display(validation_results.round(4))

ax = validation_results.sort_values("MAE (°C)", ascending=True).plot(
    x="Model", y="MAE (°C)", kind="barh", legend=False, figsize=(10, 6)
)
ax.set_xlabel("Validation MAE (°C)")
ax.set_ylabel("")
ax.set_title("Confronto dei modelli sul validation set")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 7. Valutazione finale sul test set

Dopo aver confrontato i risultati sul validation set, valutiamo sul test i modelli selezionati. Per ogni modello rifacciamo il fit su train + validation, dato che la scelta degli iperparametri è terminata. Il test viene mantenuto separato fino a questa fase.

Nota: in un'applicazione strettamente operativa, si dovrebbe verificare che la procedura di split di `Split` sia temporale e che le feature disponibili al momento della previsione non includano informazioni future.

In [ ]:
X_trainval = pd.concat([X_train, X_val]).sort_index()
y_trainval = pd.concat([y_train, y_val]).sort_index()

test_rows = []
test_predictions = {}

# Ricostruisci i modelli scelti usando gli iperparametri selezionati sul validation set.
final_models = {"Linear Regression": LinearRegression()}

final_models["Decision Tree"] = DecisionTreeRegressor(random_state=42, **best_tree_params)

for name in ["Bagging", "Random Forest", "Extra Trees"]:
    final_models[name] = make_ensemble(name, best_ensemble_params[name])

for name in ["AdaBoost", "Gradient Boosting", "HistGradientBoosting"]:
    final_models[name] = make_boosting(name, best_boosting_params[name])

for name, model in final_models.items():
    model.fit(X_trainval, y_trainval)
    pred = model.predict(X_test)
    test_predictions[name] = pred
    test_rows.append({"Model": name, **regression_metrics(y_test, pred)})

test_results = pd.DataFrame(test_rows).sort_values("MAE (°C)").reset_index(drop=True)
display(test_results.round(4))

## 8. Visualizzazione del miglior modello

Il modello mostrato qui è quello con MAE minimo sul validation set, non quello selezionato guardando il test. Il grafico confronta le previsioni con i valori reali nell'ultimo anno presente nel test set.

In [ ]:
best_model_name = validation_results.iloc[0]["Model"]
best_test_pred = test_predictions[best_model_name]

plot_df = pd.DataFrame({
    "Actual": y_test,
    "Predicted": best_test_pred,
}, index=y_test.index).sort_index()
plot_df.index = plot_df.index + pd.Timedelta(days=h)
last_year = plot_df.loc[str(plot_df.index.year.max())]

fig, axes = plt.subplots(1, 2, figsize=(15, 5), gridspec_kw={"width_ratios": [3, 1]})

axes[0].plot(last_year.index, last_year["Actual"], label="Actual", linewidth=1)
axes[0].plot(last_year.index, last_year["Predicted"], label="Predicted", linewidth=1.4)
axes[0].set_title(f"{best_model_name} — forecast a {h} giorno/i")
axes[0].set_ylabel("Temperatura media (°C)")
axes[0].legend()
axes[0].grid(alpha=0.25)

lims = [
    min(plot_df["Actual"].min(), plot_df["Predicted"].min()),
    max(plot_df["Actual"].max(), plot_df["Predicted"].max()),
]
axes[1].scatter(plot_df["Actual"], plot_df["Predicted"], s=10, alpha=0.4)
axes[1].plot(lims, lims, "--", linewidth=1)
axes[1].set_xlim(lims)
axes[1].set_ylim(lims)
axes[1].set_aspect("equal", adjustable="box")
axes[1].set_xlabel("Actual (°C)")
axes[1].set_ylabel("Predicted (°C)")
axes[1].set_title(f"Test MAE: {mean_absolute_error(y_test, best_test_pred):.2f} °C")
axes[1].grid(alpha=0.25)

plt.tight_layout()
plt.show()

print("Best model based on validation MAE:", best_model_name)
print("Test metrics:", regression_metrics(y_test, best_test_pred))

## 9. Feature importance

Gli alberi e gli ensemble ad albero espongono spesso `feature_importances_`, una misura basata sulla riduzione dell'impurità. È utile per una prima analisi, ma può essere distorta da feature correlate o da variabili con molti possibili split; non va interpretata come causalità. La permutation importance sul validation set è un'alternativa utile da valutare.

In [ ]:
importance_model_names = [
    name for name in ["Decision Tree", "Bagging", "Random Forest", "Extra Trees",
                      "AdaBoost", "Gradient Boosting", "HistGradientBoosting"]
    if hasattr(models[name], "feature_importances_")
]

fig, axes = plt.subplots(
    len(importance_model_names), 1,
    figsize=(10, 4 * len(importance_model_names)),
    squeeze=False
)

for ax, name in zip(axes[:, 0], importance_model_names):
    importances = pd.Series(models[name].feature_importances_, index=features).sort_values().tail(12)
    importances.plot(kind="barh", ax=ax)
    ax.set_title(f"Feature importance — {name}")
    ax.set_xlabel("Importance")

plt.tight_layout()
plt.show()

## 10. Interpretazione e conclusioni

Completa questa sezione dopo aver eseguito il notebook:

1. Quale modello ha il MAE più basso sul validation set? E sul test?
2. Il miglior ensemble supera il singolo Decision Tree? Se sì, osserva soprattutto la stabilità dell'errore.
3. Il boosting migliora rispetto al bagging? Non è garantito: dipende dal dataset e dagli iperparametri.
4. Ci sono differenze tra MAE e RMSE che suggeriscono errori occasionali molto grandi?
5. Quali feature risultano importanti? Lag e medie mobili possono catturare persistenza temporale, mentre `doy_sin` e `doy_cos` rappresentano la stagionalità.

**Attenzione al leakage temporale:** le feature contemporanee (ad esempio precipitazioni o temperatura del suolo del giorno `t`) sono utilizzabili solo se sono effettivamente note quando si emette la previsione per `t+h`. Se l'obiettivo è una previsione operativa reale, è necessario costruire le feature usando esclusivamente dati disponibili al tempo di emissione.